# Case Study 1: Evolution of Netflix's Content Catalog

## Introduction
This case study examines how Netflix's content catalog has changed over time by analyzing the distribution of movies, TV shows, genres, and content ratings. Using the Netflix Movies and Series dataset from Kaggle, three visualizations are developed to identify patterns in content additions and catalog composition. The study investigates changes in the number of titles added each year, the representation of popular genres over time, and differences in content ratings between movies and television shows. These visualizations provide an overview of how Netflix's catalog is structured and how its composition has evolved.

## Research Background
Ali et al. (2025), Data-Driven Insights: Machine Learning Approaches for Netflix Content Analysis and Visualization.

This paper explores Netflix's content library using Python and Tableau to create several types of visualizations, including pie charts, line graphs, histograms, and treemaps. The researchers use these visualizations to compare content across countries, analyze movie and TV show ratings, identify common genres, and track how Netflix's catalog has changed over time. One major finding was that the United States and India contributed the largest portions of content, while TV-MA was one of the most common ratings. A line graph also showed a major increase in Netflix content additions between 2014 and 2020.

This study is especially relevant to our project because it explores many of the same questions we want to answer about Netflix's catalog. Its use of line graphs to show changes over time and charts to compare countries and genres provides a useful starting point for our visualization designs. We can build on these ideas by adding interactive filters for year, genre, content type, and rating, allowing users to explore patterns and compare different parts of Netflix's catalog more easily.

We aim to expand on the paper's descriptive analysis methods by using interactive Plotly visualizations that allow users to inspect content trends and compare categories.

## Data
We are using a cleaned version of the Netflix Movies and Series dataset sourced from Kaggle.
https://www.kaggle.com/datasets/debayank2024/netflix-movies-and-series

All data cleaning was performed in src/clean-data.py as a means to ensure visualizations are accurate and relevant. Changes made are described in comments within the same file and were made to address any accuracy or aggregation concerns.

For this case study we will focus on the following fields: show_id, type, date_added, year_added, list_in, and rating

In [33]:
import pandas as pd
import plotly.express as px
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data" / "netflix_cleaned.csv").exists():
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "netflix_cleaned.csv")

df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_number,duration_minutes,seasons
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,90,90.0,NaN
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,2,NaN,2.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,1,NaN,1.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,1,NaN,1.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,2,NaN,2.0


## Visualization 1: Movies vs. TV Shows Over Time
Research question:
How has the number of movies and TV shows added to Netflix changed over time?

**Visualization Scope:** Although the dataset contains records of titles added before 2016, these years have relatively few entries compared with later periods. To make the trends easier to interpret, this visualization focuses on titles added from 2016 onward. Earlier records are retained in the dataset but excluded from this particular view. This allows for a clearer comparison of movie and TV show additions during a period with greater data representation.

In [35]:
yearly_counts = (
    df.dropna(subset=["year_added"])
      .query("year_added >= 2016")
      .groupby(["year_added", "type"])
      .size()
      .reset_index(name="count")
)

fig = px.line(
    yearly_counts,
    x="year_added",
    y="count",
    color="type",
    markers=True,
    title="Netflix Movies and TV Shows Added (2016–2021)",
    labels={
        "year_added": "Year Added",
        "count": "Number of Titles",
        "type": "Content Type"
    }
)

fig.show()

## Analysis

Based on the visualization, movies appear to have a higher number of annual additions than TV shows, with this pattern remaining consistent from 2016 to 2021. Both content types experienced their largest increases in 2019, followed by a gradual decline in the number of titles added annually. This suggests that movies and TV shows followed similar trends throughout the period, despite differences in their respective quantities. These findings generally align with Ali et al. (2025), who identified a significant increase in Netflix content additions between 2014 and 2020. While the decrease in annual additions after 2019 may indicate a slowdown in catalog expansion, it does not necessarily suggest that Netflix's overall catalog was shrinking, as the visualization only accounts for titles added each year rather than titles removed.

## Visualization 2: Genre Trends Over Time
Research question
How has the representation of popular Netflix genres changed over time?

**Visualization Scope:** Although the dataset contains records of titles added before 2016, these years have relatively few entries compared with later periods. To make the trends easier to interpret, this visualization focuses on titles added from 2016 onward. Earlier records are retained in the dataset but excluded from this particular view. This allows for a clearer comparison of movie and TV show additions during a period with greater data representation.

In [36]:
# Filter to titles added from 2016 onward
genre_df = df.dropna(subset=["year_added"]).copy()
genre_df = genre_df[genre_df["year_added"] >= 2016]

# Convert years to integers
genre_df["year_added"] = genre_df["year_added"].astype(int)

# Separate multiple genres into individual rows
genre_df["genre"] = genre_df["listed_in"].str.split(",")
genre_df = genre_df.explode("genre")
genre_df["genre"] = genre_df["genre"].str.strip()

# Identify the 10 most common genres
top_genres = genre_df["genre"].value_counts().head(10).index

# Keep only the top genres
genre_df = genre_df[genre_df["genre"].isin(top_genres)]

# Count titles by genre and year
genre_counts = (
    genre_df.groupby(["genre", "year_added"])
    .size()
    .unstack(fill_value=0)
)

# Create heatmap
fig = px.imshow(
    genre_counts,
    labels={
        "x": "Year Added",
        "y": "Genre",
        "color": "Number of Titles"
    },
    title="Netflix Genre Trends Over Time (2016–2021)",
    aspect="auto",
    text_auto=True,
    color_continuous_scale="Blues"
)

fig.update_layout(
    xaxis_title="Year Added",
    yaxis_title="Genre",
    height=550
)

fig.show()

## Analysis

Based on the heatmap, we can observe that International Movies, Dramas, and Comedies are consistently among the most represented genres in Netflix's catalog from 2016 to 2021. International Movies reached their highest number of additions in 2018 with 668 titles, while Dramas and Comedies peaked in 2019 with 563 and 420 titles, respectively. Although most genres experienced an increase in additions leading up to 2018–2019, several categories, including Romantic Movies and Children & Family Movies, continued growing into 2020. Following these peak years, most genres experienced a decline in annual additions by 2021. These patterns generally align with Visualization 1, which showed an overall increase in Netflix content additions leading up to 2019, followed by a decrease in subsequent years. However, differences in when individual genres peaked suggest that not all categories followed identical trends. It is also important to note that Netflix titles can belong to multiple genres, meaning a single title may contribute to several categories. Therefore, the values represent genre occurrences rather than entirely unique titles.

## Visualization 3: Content Ratings by Type
Research question
How do content-rating distributions differ between Netflix movies and TV shows?

In [37]:
rating_counts = (
    df.groupby(["type", "rating"])
    .size()
    .reset_index(name="count")
)

rating_counts["percentage"] = (
    rating_counts["count"] /
    rating_counts.groupby("type")["count"].transform("sum")
    * 100
)

fig = px.bar(
    rating_counts,
    x="type",
    y="percentage",
    color="rating",
    title="Netflix Content Ratings by Type",
    labels={
        "type": "Content Type",
        "percentage": "Percentage of Titles",
        "rating": "Content Rating"
    }
)

fig.update_layout(barmode="stack")
fig.show()

## Analysis

Based on the visualization, TV-MA is the most common content rating for both movies and TV shows, representing approximately 34% of movies and 43% of TV shows. TV-14 is the second most common rating in both categories, while R-rated content makes up a noticeable portion of movies and TV-PG represents a significant portion of TV shows. These results suggest that Netflix's catalog contains a large proportion of content intended for mature or teenage audiences, with mature-rated content being particularly prominent among TV shows. Although both content types share similar dominant ratings, their overall distributions differ, especially in the representation of family-friendly and younger-audience content. These findings align with Ali et al. (2025), who also identified TV-MA as one of the most frequently occurring ratings in Netflix's catalog. By comparing percentages rather than total counts, this visualization provides a clearer understanding of how content ratings are distributed between movies and TV shows, regardless of differences in the total number of titles in each category.

## Discussion of Findings

Together, the three visualizations provide a better understanding of how Netflix's content catalog has changed over time and how its content is distributed across genres and ratings. Figure 1 demonstrates that movies generally account for more annual additions than TV shows, although both categories follow similar trends. Content additions increased substantially leading up to 2019 before generally declining in subsequent years. Figure 2 provides additional evidence of this pattern, showing that major genres such as International Movies, Dramas, and Comedies experienced significant increases during the same period. International Movies peaked in 2018 with 668 titles, while Dramas and Comedies reached their highest counts in 2019. This suggests that increases in overall content additions were accompanied by growth across several popular genres. The prominence of movie-related genres also reflects the larger representation of movies in the catalog.

Figure 3 reveals differences that are not immediately apparent from total title counts. TV-MA represents approximately 34% of movies and 43% of TV shows, indicating that mature-rated content makes up a substantial portion of both categories. These findings generally align with Ali et al. (2025), who identified significant growth in Netflix content additions between 2014 and 2020 and highlighted the prominence of TV-MA ratings. Overall, the visualizations demonstrate how examining time, genre, and content ratings together provides a more complete understanding of Netflix's catalog composition.

## Limitations
- This dataset only looks at Netflix's catlogue up until 2021 and does not reflect current additions.
- Recorded additions do not establish how many titles remained available in any particular year.
- Genre categories overlap.
- Missing dates and other metadata can affect counts.
- The dataset does not include streaming statistics or viewership.

## LLM Usage
Model: ChatGPT
Prompt                                                          Usage
Analyze the dataset and identify necessary cleaning             To clean data and account for any missing values
Reccomend visualizations to display netflix catalog trends      Chart Selection
Debug and clean the base code to improve visualization quality  Used for the code for each visualization

## References:
Netflix Movies and Series dataset sourced from Kaggle.
https://www.kaggle.com/datasets/debayank2024/netflix-movies-and-series

Ali et al. (2025), Data-Driven Insights: Machine Learning Approaches for Netflix Content Analysis and Visualization.